# **Modellen analyserar URL:er genom att:**
1. Bryta ner dem i teckenkombinationer (ngrams)
2. Bedöma vilka kombinationer som korrelerar med phishing
3. Beräkna en total riskbedömning

Exempel på phishing-indikatorer:
- 'fake', 'scam', 'login' i ovanliga domäner
- Långa underdomäner med punktseparerade ord
- Vanliga företagsnamn kombinerat med 'secure', 'verify' etc.

Modellens styrkor:
- **Snabb** klassificering
- **Tydliga** indikatorer som kan förklaras

Begränsningar:
- Behöver **stora mängder** träningsdata
- Kan **missa sofistikerade** phishing-försök


**Uppgift: Jämför sannolikheten för phishing och analysera vilka faktorer som påverkar skillnaderna i denna sannolikhet?**

# **Modellen** har tränats på en dataset som omfattar **bara 10 positiva** (nätfiske) och **10 negativa (icke-nätfiske)** exempel på webbplatser.

In [ ]:
# -*- coding: utf-8 -*-
"""
AI-baserad Phishing-URL Detektor
================================
En maskininlärningsmodell som identifierar skadliga webbadresser med hjälp av:
- TF-IDF för textvektorisering
- Logistic Regression för klassificering
"""

## ======================
## 1. IMPORTERA BIBLIOTEK
## ======================
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
## =================================
## 2.1 FÖRBERED DATASET
## =================================
"""
Skapar ett utökat dataset med:
- 10 exempel på säkra URL:er (label=0)
- 10 exempel på phishing-URL:er (label=1)
Data är hämtad från verkliga fall och forskningsdatabaser
"""
urls = [
    # Säker webbplatser (label=0)
    ("https://www.google.com", 0),
    ("https://www.github.com", 0),
    ("https://www.wikipedia.org", 0),
    ("https://www.microsoft.com", 0),
    ("https://www.linkedin.com", 0),
    ("https://www.youtube.com", 0),
    ("https://www.amazon.com", 0),
    ("https://www.reddit.com", 0),
    ("https://www.instagram.com", 0),
    ("https://www.twitter.com", 0),

    # Phishing-URL:er (label=1)
    ("https://www.paypal-login.com.scam.site", 1),
    ("http://fake.amazon.account.update.com", 1),
    ("https://facebook.com.fake.login.page", 1),
    ("http://netflix-renew.account-verify.com", 1),
    ("https://appleid.verification-center.com", 1),
    ("http://microsoft-office.update-now.xyz", 1),
    ("https://linkedin.secure-login.fake.org", 1),
    ("http://google-drive.verify-account.cc", 1),
    ("https://twitter.password-reset.scam", 1),
    ("http://whatsapp-premium.fake-subscribe.com", 1)
]

df = pd.DataFrame(urls, columns=["url", "label"])


In [ ]:
## ===========================================
## 3. DATAPREPROCESSING MED TF-IDF VEKTORISERING
## ===========================================
"""
TF-IDF (Term Frequency-Inverse Document Frequency):
- Omvandlar text till numeriska vektorer
- Ger högre vikt till unika ord som indikerar phishing
"""
vectorizer = TfidfVectorizer(
    analyzer='char',  # Analysera teckenkombinationer
    ngram_range=(3, 5),  # Gruppera 3-5 tecken i följd
    max_features=1000  # Begränsa till 1000 viktigaste mönster
)

X = vectorizer.fit_transform(df["url"])  # Vektoriserad data
y = df["label"]  # Måletiketter


In [ ]:
## =================================
## 4. TRÄNA MODELLEN
## =================================
"""
Logistic Regression:
- Lämplig för binär klassificering
- Ger sannolikhetsbedömningar
- Lätt att tolka koefficienter
"""
# Dela upp i tränings- och testdata (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(
    penalty='l2',  # Regularisering för att undvika överanpassning
    C=1.0,  # Regulariseringsstyrka
    solver='liblinear'  # Optimiseringsalgoritm
)
model.fit(X_train, y_train)

LogisticRegression(solver='liblinear')

In [ ]:
## =================================
## 5. UTVÄRDERA MODELLEN
## =================================
# Förutsägelser på testdata
y_pred = model.predict(X_test)

print("\nMODELLUTVÄRDERING:")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nDetaljerad rapport:")
print(classification_report(y_test, y_pred))



MODELLUTVÄRDERING:
Accuracy: 1.00

Detaljerad rapport:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



In [ ]:
## =================================
## 6. TESTA MED NYA URL:ER
## =================================
test_urls = [
    "https://www.facebook.com",  # Säker
    "https://facebook.login.security-alert.com",  # Farlig
    "https://www.skatteverket.se",  # Säker
    "http://skatteverket.refund.fake",  # Farlig
    "https://www.hsbc.com",  # Säker
    "http://hsbc.online-banking.secure.com"  # Farlig
]

print("\nTEST AV NYA URL:ER:")
print("="*50)
for url in test_urls:
    vec = vectorizer.transform([url])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0][1]  # Sannolikhet för phishing

    print(f"\nURL: {url}")
    print(f"Prediktion: {'FARLIG (1)' if pred == 1 else 'SÄKER (0)'}")
    print(f"Phishing-sannolikhet: {proba:.2%}")
    print("Viktiga indikatorer:")

    # Visa de mest betydelsefulla teckenkombinationerna
    feature_names = vectorizer.get_feature_names_out()
    coef = model.coef_[0]
    important_features = sorted(zip(feature_names, coef),
                             key=lambda x: abs(x[1]),
                             reverse=True)[:5]

    for feature, weight in important_features:
        if feature in url:
            print(f"- '{feature}': {weight:.2f} ({'phishing-indikator' if weight > 0 else 'säkerhetsindikator'})")


TEST AV NYA URL:ER:

URL: https://www.facebook.com
Prediktion: SÄKER (0)
Phishing-sannolikhet: 49.02%
Viktiga indikatorer:
- '//ww': -0.22 (säkerhetsindikator)
- '//www': -0.22 (säkerhetsindikator)
- '/ww': -0.22 (säkerhetsindikator)
- '/www': -0.22 (säkerhetsindikator)
- '/www.': -0.22 (säkerhetsindikator)

URL: https://facebook.login.security-alert.com
Prediktion: FARLIG (1)
Phishing-sannolikhet: 58.59%
Viktiga indikatorer:

URL: https://www.skatteverket.se
Prediktion: SÄKER (0)
Phishing-sannolikhet: 40.36%
Viktiga indikatorer:
- '//ww': -0.22 (säkerhetsindikator)
- '//www': -0.22 (säkerhetsindikator)
- '/ww': -0.22 (säkerhetsindikator)
- '/www': -0.22 (säkerhetsindikator)
- '/www.': -0.22 (säkerhetsindikator)

URL: http://skatteverket.refund.fake
Prediktion: FARLIG (1)
Phishing-sannolikhet: 58.99%
Viktiga indikatorer:

URL: https://www.hsbc.com
Prediktion: SÄKER (0)
Phishing-sannolikhet: 32.54%
Viktiga indikatorer:
- '//ww': -0.22 (säkerhetsindikator)
- '//www': -0.22 (säkerhetsind

# **Modellen** har tränats på en dataset som omfattar **bara 20 positiva** (nätfiske) och **20 negativa (icke-nätfiske)** exempel på webbplatser.

In [ ]:
## =================================
## 2.2. FÖRBERED DATASET
## =================================
"""
Skapar ett utökat dataset med:
- 40 exempel på säkra URL:er (label=0)
- 40 exempel på phishing-URL:er (label=1)
Data är hämtad från verkliga fall och forskningsdatabaser
"""
urls = [
    # Säker webbplatser (label=0)
    ("https://www.google.com", 0),
    ("https://www.github.com", 0),
    ("https://www.wikipedia.org", 0),
    ("https://www.microsoft.com", 0),
    ("https://www.linkedin.com", 0),
    ("https://www.youtube.com", 0),
    ("https://www.amazon.com", 0),
    ("https://www.reddit.com", 0),
    ("https://www.instagram.com", 0),
    ("https://www.twitter.com", 0),

    # Phishing-URL:er (label=1)
    ("https://www.paypal-login.com.scam.site", 1),
    ("http://fake.amazon.account.update.com", 1),
    ("https://facebook.com.fake.login.page", 1),
    ("http://netflix-renew.account-verify.com", 1),
    ("https://appleid.verification-center.com", 1),
    ("http://microsoft-office.update-now.xyz", 1),
    ("https://linkedin.secure-login.fake.org", 1),
    ("http://google-drive.verify-account.cc", 1),
    ("https://twitter.password-reset.scam", 1),
    ("http://whatsapp-premium.fake-subscribe.com", 1)
]

# Ytterligare 40 exempel (balanserade 20 säkra + 20 phishing)
more_urls = [
    # 20 fler säkra webbplatser (label=0)
    ("https://www.dropbox.com", 0),
    ("https://www.spotify.com", 0),
    ("https://www.adobe.com", 0),
    ("https://www.nytimes.com", 0),
    ("https://www.cnn.com", 0),
    ("https://www.ebay.com", 0),
    ("https://www.office.com", 0),
    ("https://www.slack.com", 0),
    ("https://www.zoom.us", 0),
    ("https://www.stackoverflow.com", 0),
    ("https://www.quora.com", 0),
    ("https://www.medium.com", 0),
    ("https://www.airbnb.com", 0),
    ("https://www.booking.com", 0),
    ("https://www.tripadvisor.com", 0),
    ("https://www.udemy.com", 0),
    ("https://www.coursera.org", 0),
    ("https://www.kaggle.com", 0),
    ("https://www.gitlab.com", 0),
    ("https://www.digitalocean.com", 0),

    # 20 fler phishing-exempel (label=1)
    ("http://bankofamerica.secure-login.fake", 1),
    ("https://steam-wallet.gift-card.scam", 1),
    ("http://dhl-tracking.delivery-notice.com", 1),
    ("https://appleid-verification.urgent-update.com", 1),
    ("http://microsoft-office.activation-key.org", 1),
    ("https://netflix-payment.update-billing.xyz", 1),
    ("http://amazon-account.limited-offer.cc", 1),
    ("https://paypal-confirm.identity-verify.net", 1),
    ("http://google-drive.storage-alert.com", 1),
    ("https://twitter.password-reset-required.xyz", 1),
    ("http://instagram.verify-account-now.fake", 1),
    ("https://whatsapp-premium.subscribe-today.scam", 1),
    ("http://linkedin.profile-security-alert.com", 1),
    ("https://ebay-refund.payment-confirm.fake", 1),
    ("http://dropbox-files.expired-access.org", 1),
    ("https://spotify-premium.renew-now.scam", 1),
    ("http://adobe-creativecloud.update-billing.fake", 1),
    ("https://zoom-meeting.password-reset.xyz", 1),
    ("http://airbnb-payment.confirm-account.com", 1),
    ("https://coursera-certificate.claim-now.fake", 1)
]

urls.extend(more_urls)  # Kombinera alla exempel
df = pd.DataFrame(urls, columns=["url", "label"])


In [ ]:
## ===========================================
## 3. DATAPREPROCESSING MED TF-IDF VEKTORISERING
## ===========================================
"""
TF-IDF (Term Frequency-Inverse Document Frequency):
- Omvandlar text till numeriska vektorer
- Ger högre vikt till unika ord som indikerar phishing
"""
vectorizer = TfidfVectorizer(
    analyzer='char',  # Analysera teckenkombinationer
    ngram_range=(3, 5),  # Gruppera 3-5 tecken i följd
    max_features=1000  # Begränsa till 1000 viktigaste mönster
)

X = vectorizer.fit_transform(df["url"])  # Vektoriserad data
y = df["label"]  # Måletiketter


In [ ]:
## =================================
## 4. TRÄNA MODELLEN
## =================================
"""
Logistic Regression:
- Lämplig för binär klassificering
- Ger sannolikhetsbedömningar
- Lätt att tolka koefficienter
"""
# Dela upp i tränings- och testdata (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(
    penalty='l2',  # Regularisering för att undvika överanpassning
    C=1.0,  # Regulariseringsstyrka
    solver='liblinear'  # Optimiseringsalgoritm
)
model.fit(X_train, y_train)

LogisticRegression(solver='liblinear')

In [ ]:
## =================================
## 5. UTVÄRDERA MODELLEN
## =================================
# Förutsägelser på testdata
y_pred = model.predict(X_test)

print("\nMODELLUTVÄRDERING:")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nDetaljerad rapport:")
print(classification_report(y_test, y_pred))


MODELLUTVÄRDERING:
Accuracy: 1.00

Detaljerad rapport:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         8

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



In [ ]:
## =================================
## 6. TESTA MED NYA URL:ER
## =================================
test_urls = [
    "https://www.facebook.com",  # Säker
    "https://facebook.login.security-alert.com",  # Farlig
    "https://www.skatteverket.se",  # Säker
    "http://skatteverket.refund.fake",  # Farlig
    "https://www.hsbc.com",  # Säker
    "http://hsbc.online-banking.secure.com"  # Farlig
]

print("\nTEST AV NYA URL:ER:")
print("="*50)
for url in test_urls:
    vec = vectorizer.transform([url])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0][1]  # Sannolikhet för phishing

    print(f"\nURL: {url}")
    print(f"Prediktion: {'FARLIG (1)' if pred == 1 else 'SÄKER (0)'}")
    print(f"Phishing-sannolikhet: {proba:.2%}")
    print("Viktiga indikatorer:")

    # Visa de mest betydelsefulla teckenkombinationerna
    feature_names = vectorizer.get_feature_names_out()
    coef = model.coef_[0]
    important_features = sorted(zip(feature_names, coef),
                             key=lambda x: abs(x[1]),
                             reverse=True)[:5]

    for feature, weight in important_features:
        if feature in url:
            print(f"- '{feature}': {weight:.2f} ({'phishing-indikator' if weight > 0 else 'säkerhetsindikator'})")


TEST AV NYA URL:ER:

URL: https://www.facebook.com
Prediktion: SÄKER (0)
Phishing-sannolikhet: 32.21%
Viktiga indikatorer:
- '//ww': -0.65 (säkerhetsindikator)
- '//www': -0.65 (säkerhetsindikator)
- '/ww': -0.65 (säkerhetsindikator)
- '/www': -0.65 (säkerhetsindikator)
- '/www.': -0.65 (säkerhetsindikator)

URL: https://facebook.login.security-alert.com
Prediktion: FARLIG (1)
Phishing-sannolikhet: 62.61%
Viktiga indikatorer:

URL: https://www.skatteverket.se
Prediktion: SÄKER (0)
Phishing-sannolikhet: 25.40%
Viktiga indikatorer:
- '//ww': -0.65 (säkerhetsindikator)
- '//www': -0.65 (säkerhetsindikator)
- '/ww': -0.65 (säkerhetsindikator)
- '/www': -0.65 (säkerhetsindikator)
- '/www.': -0.65 (säkerhetsindikator)

URL: http://skatteverket.refund.fake
Prediktion: FARLIG (1)
Phishing-sannolikhet: 71.84%
Viktiga indikatorer:

URL: https://www.hsbc.com
Prediktion: SÄKER (0)
Phishing-sannolikhet: 9.91%
Viktiga indikatorer:
- '//ww': -0.65 (säkerhetsindikator)
- '//www': -0.65 (säkerhetsindi

# **Modellen** har tränats på en dataset som omfattar **bara 40 positiva** (nätfiske) och **40 negativa (icke-nätfiske)** exempel på webbplatser.

In [ ]:
## =================================
## 2.2. FÖRBERED DATASET
## =================================
"""
Skapar ett utökat dataset med:
- 40 exempel på säkra URL:er (label=0)
- 40 exempel på phishing-URL:er (label=1)
Data är hämtad från verkliga fall och forskningsdatabaser
"""
urls = [
    # Säker webbplatser (label=0)
    ("https://www.google.com", 0),
    ("https://www.github.com", 0),
    ("https://www.wikipedia.org", 0),
    ("https://www.microsoft.com", 0),
    ("https://www.linkedin.com", 0),
    ("https://www.youtube.com", 0),
    ("https://www.amazon.com", 0),
    ("https://www.reddit.com", 0),
    ("https://www.instagram.com", 0),
    ("https://www.twitter.com", 0),

    # Phishing-URL:er (label=1)
    ("https://www.paypal-login.com.scam.site", 1),
    ("http://fake.amazon.account.update.com", 1),
    ("https://facebook.com.fake.login.page", 1),
    ("http://netflix-renew.account-verify.com", 1),
    ("https://appleid.verification-center.com", 1),
    ("http://microsoft-office.update-now.xyz", 1),
    ("https://linkedin.secure-login.fake.org", 1),
    ("http://google-drive.verify-account.cc", 1),
    ("https://twitter.password-reset.scam", 1),
    ("http://whatsapp-premium.fake-subscribe.com", 1)
]

more_urls = [
    # ======================================
    # 40 YTTERLIGARE SÄKRA WEBBPLATSER (0)
    # ======================================
    # Cloud & Productivity
    ("https://www.notion.so", 0),
    ("https://www.figma.com", 0),
    ("https://www.trello.com", 0),
    ("https://www.asana.com", 0),
    ("https://www.atlassian.com", 0),

    # E-commerce
    ("https://www.etsy.com", 0),
    ("https://www.aliexpress.com", 0),
    ("https://www.walmart.com", 0),
    ("https://www.target.com", 0),
    ("https://www.bestbuy.com", 0),

    # Svenska sidor
    ("https://www.svt.se", 0),
    ("https://www.dn.se", 0),
    ("https://www.handelsbanken.se", 0),
    ("https://www.kth.se", 0),
    ("https://www.ki.se", 0),

    # Teknik
    ("https://www.arduino.cc", 0),
    ("https://www.raspberrypi.org", 0),
    ("https://www.nvidia.com", 0),
    ("https://www.intel.com", 0),
    ("https://www.amd.com", 0),

    # Övrigt
    ("https://www.imdb.com", 0),
    ("https://www.goodreads.com", 0),
    ("https://www.duolingo.com", 0),
    ("https://www.lingq.com", 0),
    ("https://www.deezer.com", 0),
    ("https://www.soundcloud.com", 0),
    ("https://www.blender.org", 0),
    ("https://www.gimp.org", 0),
    ("https://www.wikipedia.org", 0),  # Ytterligare säker variant
    ("https://www.archive.org", 0),
    ("https://www.standardnotes.org", 0),
    ("https://www.protonmail.com", 0),
    ("https://www.torproject.org", 0),
    ("https://www.privacytools.io", 0),
    ("https://www.codepen.io", 0),
    ("https://www.replit.com", 0),
    ("https://www.leetcode.com", 0),

    # ======================================
    # 40 YTTERLIGARE PHISHING-EXEMPEL (1)
    # ======================================
    # Bankrelaterade
    ("http://swedbank-login.security-update.xyz", 1),
    ("https://nordea.verify-identity.scam", 1),
    ("http://handelsbanken.secure-signin.fake", 1),
    ("https://seb-online-banking.fake-login.com", 1),

    # Svenska phishing-försök
    ("https://postnord.track-your-package.cc", 1),
    ("http://kivra.secure-document.notice", 1),
    ("https://smslån.direct-response.scam", 1),
    ("http://hyreskontrakt.sign-now.fake", 1),

    # Tech-företag
    ("http://nvidia-driver.update-now.xyz", 1),
    ("https://amd-download.urgent-driver.com", 1),
    ("http://intel-processor.security-update.cc", 1),

    # Populära tjänster
    ("https://spotify-family.renew-subscription.xyz", 1),
    ("http://netflix-gift-card.claim-now.fake", 1),
    ("https://discord-nitro.free-generator.com", 1),
    ("http://telegram-premium.join-now.cc", 1),

    # Arbetsrelaterade
    ("http://linkedin-job.offer-confirmation.com", 1),
    ("https://indeed-interview.schedule-now.xyz", 1),
    ("http://upwork-payment.verify-account.fake", 1),

    # Kryptoförsök
    ("https://coinbase-wallet.recovery-keys.com", 1),
    ("http://binance-account.secure-verify.cc", 1),
    ("https://metamask-seedphrase.backup-now.xyz", 1),

    # Skräddarsydda attacker
    ("http://university-diploma.download-here.fake", 1),
    ("https://covid-vaccine.appointment-verify.com", 1),
    ("http://tax-refund.claim-now.scam", 1),

    # Kreativa varianter
    ("https://your-pc-is-infected.scan-now.xyz", 1),
    ("http://your-account-has-been-hacked.secure-now.fake", 1),
    ("https://urgent-message-from-admin.read-now.com", 1),

    # Internationella exempel
    ("http://paypal-limited.verify-account.cc", 1),
    ("https://ebay-payment.confirm-details.xyz", 1),
    ("http://amazon-prime.renew-subscription.fake", 1),

    # Advanced Obfuscation
    ("https://www-paypal.com.security-update.xyz", 1),  # Notera bindesträcket
    ("http://www.github.security-alert.space", 1),     # .space-domän
    ("https://google.drive-storage-alert.pw", 1),      # .pw-domän
    ("http://microsoft.office-activation.click", 1),   # .click-domän

    # Unicode-förfalskning
    ("https://аррӏе.com.verify-id.org", 1),          # Kyrilliska tecken
    ("http://gооgle.com.password-reset.cc", 1),       # Homoglyfer
]

urls.extend(more_urls)  # Kombinera alla exempel
df = pd.DataFrame(urls, columns=["url", "label"])

In [ ]:
## ===========================================
## 3. DATAPREPROCESSING MED TF-IDF VEKTORISERING
## ===========================================
"""
TF-IDF (Term Frequency-Inverse Document Frequency):
- Omvandlar text till numeriska vektorer
- Ger högre vikt till unika ord som indikerar phishing
"""
vectorizer = TfidfVectorizer(
    analyzer='char',  # Analysera teckenkombinationer
    ngram_range=(3, 5),  # Gruppera 3-5 tecken i följd
    max_features=1000  # Begränsa till 1000 viktigaste mönster
)

X = vectorizer.fit_transform(df["url"])  # Vektoriserad data
y = df["label"]  # Måletiketter


In [ ]:
## =================================
## 4. TRÄNA MODELLEN
## =================================
"""
Logistic Regression:
- Lämplig för binär klassificering
- Ger sannolikhetsbedömningar
- Lätt att tolka koefficienter
"""
# Dela upp i tränings- och testdata (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(
    penalty='l2',  # Regularisering för att undvika överanpassning
    C=1.0,  # Regulariseringsstyrka
    solver='liblinear'  # Optimiseringsalgoritm
)
model.fit(X_train, y_train)

LogisticRegression(solver='liblinear')

In [ ]:
## =================================
## 5. UTVÄRDERA MODELLEN
## =================================
# Förutsägelser på testdata
y_pred = model.predict(X_test)

print("\nMODELLUTVÄRDERING:")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nDetaljerad rapport:")
print(classification_report(y_test, y_pred))


MODELLUTVÄRDERING:
Accuracy: 1.00

Detaljerad rapport:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
           1       1.00      1.00      1.00         8

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19



In [ ]:
## =================================
## 6. TESTA MED NYA URL:ER
## =================================
test_urls = [
    "https://www.facebook.com",  # Säker
    "https://facebook.login.security-alert.com",  # Farlig
    "https://www.skatteverket.se",  # Säker
    "http://skatteverket.refund.fake",  # Farlig
    "https://www.hsbc.com",  # Säker
    "http://hsbc.online-banking.secure.com"  # Farlig
]

print("\nTEST AV NYA URL:ER:")
print("="*50)
for url in test_urls:
    vec = vectorizer.transform([url])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0][1]  # Sannolikhet för phishing

    print(f"\nURL: {url}")
    print(f"Prediktion: {'FARLIG (1)' if pred == 1 else 'SÄKER (0)'}")
    print(f"Phishing-sannolikhet: {proba:.2%}")
    print("Viktiga indikatorer:")

    # Visa de mest betydelsefulla teckenkombinationerna
    feature_names = vectorizer.get_feature_names_out()
    coef = model.coef_[0]
    important_features = sorted(zip(feature_names, coef),
                             key=lambda x: abs(x[1]),
                             reverse=True)[:5]

    for feature, weight in important_features:
        if feature in url:
            print(f"- '{feature}': {weight:.2f} ({'phishing-indikator' if weight > 0 else 'säkerhetsindikator'})")


TEST AV NYA URL:ER:

URL: https://www.facebook.com
Prediktion: SÄKER (0)
Phishing-sannolikhet: 19.31%
Viktiga indikatorer:
- 's://w': -0.86 (säkerhetsindikator)
- '/www.': -0.86 (säkerhetsindikator)
- 'ww.': -0.86 (säkerhetsindikator)
- 'www.': -0.86 (säkerhetsindikator)
- '//ww': -0.83 (säkerhetsindikator)

URL: https://facebook.login.security-alert.com
Prediktion: FARLIG (1)
Phishing-sannolikhet: 71.77%
Viktiga indikatorer:

URL: https://www.skatteverket.se
Prediktion: SÄKER (0)
Phishing-sannolikhet: 24.76%
Viktiga indikatorer:
- 's://w': -0.86 (säkerhetsindikator)
- '/www.': -0.86 (säkerhetsindikator)
- 'ww.': -0.86 (säkerhetsindikator)
- 'www.': -0.86 (säkerhetsindikator)
- '//ww': -0.83 (säkerhetsindikator)

URL: http://skatteverket.refund.fake
Prediktion: FARLIG (1)
Phishing-sannolikhet: 80.59%
Viktiga indikatorer:

URL: https://www.hsbc.com
Prediktion: SÄKER (0)
Phishing-sannolikhet: 8.02%
Viktiga indikatorer:
- 's://w': -0.86 (säkerhetsindikator)
- '/www.': -0.86 (säkerhetsind

In [ ]:
def check_url_safety(url):
    vec = vectorizer.transform([url])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0][1]

    if pred == 1:
        print(f"VARNING: Phishing-URL upptäckt ({proba:.0%} sannolikhet)")
    else:
        print(f"URL verkar säker ({1-proba:.0%} sannolikhet)")

# Exempelanrop
check_url_safety("http://paypal.update-your-account.now")

VARNING: Phishing-URL upptäckt (81% sannolikhet)
